# Chapter 5 — From LLM to AI Agent

By now you should understand an important fact:
> **A raw LLM by itself cannot use tools, cannot remember anything, and cannot perform multi-step reasoning in the real world.**

A "powerful AI agent" is **not just the LLM**. Instead, it is **software that coordinates multiple components around the LLM**: tools (search, calculator, database), memory, planning logic, and interaction loops. The orchestration software decides the behavior of the agent during each interaction.

## Table of Contents

1. [Introduction: Single AI Agent and Multi-Agent Systems](#51-single-agent-and-multi-agent-system)
2. [Overview: Components of an AI Agent](#52-overview-components-of-an-ai-agent)
3. [Example 1: Basic Agent Without Memory](#53-example-1-basic-agent-without-memory)
4. [Example 2: Agent With Checkpoint and Context](#54-example-2-agent-with-checkpoint-and-context)
5. [Example 3: Understanding Recursion Limit](#55-example-3-understanding-recursion-limit)
6. [Summary](#56-summary)

## 5.1 Single Agent and Multi-Agent System

### A Human Analogy for a Single Agent

Think of single AI agents as **individual humans**. A person alone has limited capability, but when we give that person tools, their abilities expand dramatically:

| Tool | New Ability |
|-----|-------------|
| Camera | Capture images |
| Laptop | Write software |
| Internet | Access global knowledge |
| Hard drive | Store long-term memory |

The same idea applies to AI systems:

```
LLM + calculator tool → can perform reliable math
LLM + search tool → can access external knowledge
LLM + database → can remember information
```

### From Individual to Organization

You might often hear about multi-agent systems. Think of this as **multiple humans working together**: researchers, analysts, engineers, managers. When coordination works well, a **team becomes far more powerful than any individual**. The same principle leads to **multi-agent systems**.

However, just like human organizations, once many agents are involved we encounter new challenges: communication overhead, conflicting goals, coordination complexity, task scheduling, and information sharing. For example:

- Should messages be **broadcast to all agents**, or only to specific ones?
- Which tasks should be **parallelized**, and which must remain sequential?
- How should agents **debate or verify each other's conclusions**?

Designing these coordination patterns becomes an **important part of AI system architecture**.

### Simple Agents vs Complex Systems

For simple tasks, implementing an agent yourself is surprisingly easy. A minimal agent loop can be written in just a few minutes.

However, for larger systems you typically have two choices:

1. **Design your own framework** that coordinates tools, memory, and agents
2. **Adopt an existing framework**

In this tutorial we will focus on **LangChain and LangGraph**, but the concepts are the same for other frameworks.

### `create_agent` in LangChain

LangChain provides a high-level API called **`create_agent`** that allows us to quickly build a working agent. Internally, this function actually builds a **LangGraph workflow**, but for now we will ignore the internal mechanism and simply learn how to use it.

**We will discuss LangGraph in detail in later chapters.**

## 5.2 Overview: Components of an AI Agent

Before building agents with `create_agent`, it is helpful to understand that an AI agent system contains **two types of components**:

1. **Structural components** — define what the agent is capable of doing
2. **Runtime configuration** — controls how the agent behaves during execution

> **Note:** This chapter introduces the fundamental concepts to help you transition from thinking about LLMs to thinking about agent systems. We highlight several key components below to give you a flavor of how agents work, but this is not an exhaustive list. As you progress through the tutorial, you'll encounter additional components and more advanced patterns.

### A Simple Analogy: A Bicycle

A useful analogy is a **bicycle**. A bicycle must have structural parts: wheels, frame, brakes, chain, and optionally gears. When these components are assembled, you have a **complete bicycle**. However, the bicycle is still **static** until someone rides it.

During the ride, you make **runtime decisions**: which gear to use, when to brake, whether to turn on the light, how fast to pedal. These runtime choices do not change the structure of the bicycle. They only affect **how it behaves while being used**.

### Applying This to AI Agents

AI agents follow a very similar pattern. First we have **structural components** (LLM, tools, etc.) that define what the agent **can do**. Then during execution we provide **runtime configuration** (thread ID, context, recursion limit, etc.) that determines **how the agent behaves during a particular interaction**.

The official LangChain documentation describes these in detail:
https://docs.langchain.com/oss/python/langchain/agents

### Structural Components

Structural components are typically specified **when the agent is created**. Here are some of the most important ones:

**LLM** — The reasoning engine of the agent. Think of the LLM as the **brain of the agent**.

```python
model = init_chat_model("openai:gpt-4.1-mini")
```

**Tools** — Functions that the agent is allowed to invoke. These extend the agent beyond pure text generation. The agent can decide when to call these tools during reasoning.

**Memory** — Allows the agent to retain information across interactions. With memory, it can behave more like a **long-term assistant**.

**Checkpoint** — Stores the intermediate execution state of the agent. It allows the system to maintain the history of interactions and continue from where it left off.

```python
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()
```

### Runtime Configuration

Runtime configuration parameters are supplied **when the agent runs**. They control the behavior of the execution process. Here are several commonly used ones:

**Thread** — A thread ID identifies a conversation session. All interactions that share the same thread belong to the same conversation state.

```python
config = {"configurable": {"thread_id": "session_1"}}
```

**Context** — Contains structured information provided by the surrounding application. This information is available to the agent during execution.

**Recursion Limit** — Controls how many reasoning or tool-calling steps the agent can take. Agent execution often follows a loop: `LLM → Tool → LLM → Tool → LLM`. Without a limit, an agent could potentially enter an infinite loop. The recursion limit acts as a **safety control**.

```python
config = {"recursion_limit": 4}
```

### Important Note

Some structural components (such as checkpoint and memory) **enable runtime features**, but they still require runtime information to operate correctly. For example, even if a checkpoint system is defined when creating the agent, the system still needs a **thread ID at runtime** to determine which conversation state should be loaded.

In other words: **Agent creation defines capabilities. Agent invocation defines behavior.**

Understanding this distinction will help you design more complex agent systems later in this tutorial.

## 5.3 Example 1: Basic Agent Without Memory

We start with the smallest useful agent: one **LLM**, two **tools**, with **no checkpoint**, **no context**, and **no memory**.

This agent can recommend a shirt from a small product list. If the user does not provide any preference, it simply chooses one product randomly.

This example demonstrates the basic agent loop: the LLM calls tools, receives results, and generates a response.

In [4]:
# Minimal LangChain Agent Example
# Structural components: LLM + Tools
# No checkpoint, no context, no memory

import os
import random
from typing import Optional, List, Dict, Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


# ----------------------------
# Model configuration (LM Studio)
# ----------------------------

base_url = os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1")
api_key = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,

)


# ----------------------------
# Product database
# ----------------------------

PRODUCTS = [
    {"id": 1, "name": "Classic Red Shirt", "color": "red", "style": "casual"},
    {"id": 2, "name": "Formal Blue Shirt", "color": "blue", "style": "formal"},
    {"id": 3, "name": "Relaxed Green Shirt", "color": "green", "style": "casual"},
    {"id": 4, "name": "Minimal White Shirt", "color": "white", "style": "formal"},
    {"id": 5, "name": "Bright Red Polo", "color": "red", "style": "sport"},
]


# ----------------------------
# Tools
# ----------------------------

def list_products(color: Optional[str] = None, style: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Return products filtered by color or style.
    If no filter is given, return all products.
    """
    results = PRODUCTS

    if color:
        results = [p for p in results if p["color"].lower() == color.lower()]

    if style:
        results = [p for p in results if p["style"].lower() == style.lower()]

    return results


def pick_product(products: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Pick one product randomly from the candidate list.
    """
    if not products:
        return {"error": "No matching product found"}

    return random.choice(products)


# ----------------------------
# Create the minimal agent
# ----------------------------

agent = create_agent(
    model=model,
    tools=[list_products, pick_product],
    system_prompt=(
        "You are a simple shopping assistant.\n"
        "Help the user choose a shirt from the product list.\n"
        "If the user provides a preference such as color or style, filter the products.\n"
        "If the user provides no preference, choose randomly.\n"
        "Use the available tools to perform the task."
    ),
)


# ----------------------------
# Run the agent
# ----------------------------

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Recommend me a shirt."}
        ]
    }
)

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Recommend me a shirt.
================================== Ai Message ==================================
Tool Calls:
  list_products (112533780)
 Call ID: 112533780
  Args:
    color: None
    style: None
================================= Tool Message =================================
Name: list_products

[{"id": 1, "name": "Classic Red Shirt", "color": "red", "style": "casual"}, {"id": 2, "name": "Formal Blue Shirt", "color": "blue", "style": "formal"}, {"id": 3, "name": "Relaxed Green Shirt", "color": "green", "style": "casual"}, {"id": 4, "name": "Minimal White Shirt", "color": "white", "style": "formal"}, {"id": 5, "name": "Bright Red Polo", "color": "red", "style": "sport"}]
================================== Ai Message ==================================
Tool Calls:
  pick_product (333415627)
 Call ID: 333415627
  Args:
    products: [{'id': 1, 'name': 'Classic Red Shirt', 'color': 'red', 'style': 'cas

### Observation: The Agent Does Not Remember

From the above results, you can observe the communication between the human, the LLM, and the tools, until the system produces the final recommendation.

Now let us observe the same agent in a chatbot setting. We will simulate **two conversation turns**. Pay attention to whether this agent keeps any information from the previous conversation.

Since this agent was created without checkpoint, memory, or context, each call is effectively independent.

In [6]:
# Two-turn chatbot simulation with the same minimal agent
# Observe whether the agent remembers anything from the first turn.

turn_1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Recommend me one shirt."}
        ]
    }
)

print("=== Turn 1 ===")
for m in turn_1["messages"]:
    m.pretty_print()

print("\n" + "=" * 80 + "\n")

turn_2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What did you recommend to me just now?"}
        ]
    }
)

print(">>>>> Turn 2 >>>>")
for m in turn_2["messages"]:
    m.pretty_print()

=== Turn 1 ===
================================ Human Message =================================

Recommend me one shirt.
================================== Ai Message ==================================
Tool Calls:
  list_products (167796005)
 Call ID: 167796005
  Args:
    color: None
    style: None
================================= Tool Message =================================
Name: list_products

[{"id": 1, "name": "Classic Red Shirt", "color": "red", "style": "casual"}, {"id": 2, "name": "Formal Blue Shirt", "color": "blue", "style": "formal"}, {"id": 3, "name": "Relaxed Green Shirt", "color": "green", "style": "casual"}, {"id": 4, "name": "Minimal White Shirt", "color": "white", "style": "formal"}, {"id": 5, "name": "Bright Red Polo", "color": "red", "style": "sport"}]
================================== Ai Message ==================================
Tool Calls:
  pick_product (535670707)
 Call ID: 535670707
  Args:
    products: [{'id': 1, 'name': 'Classic Red Shirt', 'color': 're

### Why the Agent Forgets

In the second turn, the agent does **not** remember the previous recommendation.

This is because the current agent only contains an LLM and tools. It does **not** contain checkpoint, memory, or runtime context. So each invocation starts from scratch.

This is an important observation: **Tool calling alone does not make an agent stateful.**

To enable the agent to remember previous interactions, we need to add a **checkpoint system** and provide a **thread ID** at runtime. Let's see how that works in the next example.


----

## 5.4 Example 2: Agent With Checkpoint and Thread

In this example, we use **exactly the same code as Example 1**, with only **two differences**:

1. We add a **checkpoint** (structural component)
2. We provide a **thread ID** (runtime configuration)

This minimal change demonstrates how checkpoint + thread ID enables conversation memory, making it easy to compare stateless vs stateful agents.

In [9]:
# Agent example with checkpoint and thread enabled
# Same code as Example 1, with checkpoint + thread added

import os
import random
from typing import Optional, List, Dict, Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver


# ----------------------------
# Model configuration (LM Studio)
# ----------------------------

base_url = os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1")
api_key = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
)


# ----------------------------
# Product database
# ----------------------------

PRODUCTS = [
    {"id": 1, "name": "Classic Red Shirt", "color": "red", "style": "casual"},
    {"id": 2, "name": "Formal Blue Shirt", "color": "blue", "style": "formal"},
    {"id": 3, "name": "Relaxed Green Shirt", "color": "green", "style": "casual"},
    {"id": 4, "name": "Minimal White Shirt", "color": "white", "style": "formal"},
    {"id": 5, "name": "Bright Red Polo", "color": "red", "style": "sport"},
]


# ----------------------------
# Tools
# ----------------------------

def list_products(color: Optional[str] = None, style: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Return products filtered by color or style.
    If no filter is given, return all products.
    """
    results = PRODUCTS

    if color:
        results = [p for p in results if p["color"].lower() == color.lower()]

    if style:
        results = [p for p in results if p["style"].lower() == style.lower()]

    return results


def pick_product(products: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Pick one product randomly from the candidate list.
    """
    if not products:
        return {"error": "No matching product found"}

    return random.choice(products)


# ----------------------------
# NEW: Checkpoint system
# ----------------------------

checkpointer = InMemorySaver()


# ----------------------------
# Create agent (with checkpoint)
# ----------------------------

agent = create_agent(
    model=model,
    tools=[list_products, pick_product],
    checkpointer=checkpointer,  # NEW: added checkpoint
    system_prompt=(
        "You are a simple shopping assistant.\n"
        "Help the user choose a shirt from the product list.\n"
        "If the user provides a preference such as color or style, filter the products.\n"
        "If the user provides no preference, choose randomly.\n"
        "Use the available tools to perform the task."
    ),
)


# ----------------------------
# Two-turn conversation simulation
# Same test as Example 1, but with thread ID
# ----------------------------

turn_1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Recommend me one shirt."}
        ]
    },
    config={"configurable": {"thread_id": "demo-thread"}}  # NEW: added thread ID
)

print("=== Turn 1 ===")
for m in turn_1["messages"]:
    m.pretty_print()

print("\n" + "=" * 80 + "\n")

turn_2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What did you recommend to me just now?"}
        ]
    },
    config={"configurable": {"thread_id": "demo-thread"}}  # NEW: same thread ID
)

print(">>>>> Turn 2 >>>>")
for m in turn_2["messages"]:
    m.pretty_print()

=== Turn 1 ===
================================ Human Message =================================

Recommend me one shirt.
================================== Ai Message ==================================
Tool Calls:
  list_products (282891937)
 Call ID: 282891937
  Args:
    color: None
    style: None
================================= Tool Message =================================
Name: list_products

[{"id": 1, "name": "Classic Red Shirt", "color": "red", "style": "casual"}, {"id": 2, "name": "Formal Blue Shirt", "color": "blue", "style": "formal"}, {"id": 3, "name": "Relaxed Green Shirt", "color": "green", "style": "casual"}, {"id": 4, "name": "Minimal White Shirt", "color": "white", "style": "formal"}, {"id": 5, "name": "Bright Red Polo", "color": "red", "style": "sport"}]
================================== Ai Message ==================================
Tool Calls:
  pick_product (312264386)
 Call ID: 312264386
  Args:
    products: [{'id': 1, 'name': 'Classic Red Shirt', 'color': 're

### Observation: Now the Agent Remembers

With only two small changes (checkpoint + thread ID), the agent now **remembers the previous conversation**.

**What Changed:**

1. **Added checkpoint when creating the agent:**
   ```python
   checkpointer = InMemorySaver()
   agent = create_agent(..., checkpointer=checkpointer)
   ```

2. **Added thread ID when invoking the agent:**
   ```python
   config={"configurable": {"thread_id": "demo-thread"}}
   ```

**The Result:**

In Turn 2, when asked "What did you recommend to me just now?", the agent can now recall its previous recommendation from Turn 1. The checkpoint stores the conversation history, and the thread ID tells the system which conversation to load.

**What's Happening Under the Hood:**

Notice that in Turn 2, the human only asks **one simple question**: "What did you recommend to me just now?" However, the checkpoint system automatically includes **all the conversation history from Turn 1** in the message context sent to the LLM.

This means the LLM receives:
- Turn 1's human message
- Turn 1's tool calls and responses
- Turn 1's final AI response
- Turn 2's new human message

If you're using **LM Studio**, you can verify this by checking the **Developer Logs** (Server Logs tab). You'll see that the request in Turn 2 contains the entire conversation history, not just the latest question. This is how the agent "remembers" — the checkpoint loads all previous messages and sends them as context to the LLM.

**Key Insight:**

This side-by-side comparison demonstrates a fundamental principle:
- **Example 1** (no checkpoint): Each call is independent, no memory
- **Example 2** (with checkpoint + thread): Full conversation history is maintained and sent to the LLM

The difference is just a few lines of code, but the behavior changes dramatically. This is the power of combining structural components (checkpoint) with runtime configuration (thread ID).

## 5.5 Example 3: Understanding Recursion Limit

So far we have discussed structural components (LLM, tools, checkpoint, memory) and runtime configuration (thread, context). Another important runtime parameter is the **recursion limit**.

During execution, an agent usually follows a loop like this:

```text
LLM → Tool → LLM → Tool → LLM → ...
```

Each reasoning or tool-calling step consumes part of the allowed execution budget. If the recursion limit is too small, the agent may stop before finishing the task, even if the tools and prompt are correct.

In this example, we intentionally design a task that requires several steps:

1. Call `list_all_products`
2. Call `filter_by_color` on the result
3. Call `filter_by_style` on the filtered result
4. Call `pick_product` on the final candidates
5. Return the final recommendation

We will run the same task twice: first with a **very small recursion limit** (2), then with a **larger one** (20). Observe the difference in behavior.

In [10]:
# Example 3: demonstrate recursion_limit

import os
import random
from typing import List, Dict, Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


# ----------------------------
# Model configuration (LM Studio)
# ----------------------------

base_url = os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1")
api_key = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,

)


# ----------------------------
# Product database
# ----------------------------

PRODUCTS = [
    {"id": 1, "name": "Classic Red Shirt", "color": "red", "style": "casual"},
    {"id": 2, "name": "Formal Blue Shirt", "color": "blue", "style": "formal"},
    {"id": 3, "name": "Relaxed Green Shirt", "color": "green", "style": "casual"},
    {"id": 4, "name": "Minimal White Shirt", "color": "white", "style": "formal"},
    {"id": 5, "name": "Bright Red Polo", "color": "red", "style": "sport"},
    {"id": 6, "name": "Soft Red Linen Shirt", "color": "red", "style": "casual"},
]


# ----------------------------
# Tools
# ----------------------------

def list_all_products() -> List[Dict[str, Any]]:
    """Return all available products."""
    return PRODUCTS


def filter_by_color(products: List[Dict[str, Any]], color: str) -> List[Dict[str, Any]]:
    """Filter products by color."""
    return [p for p in products if p["color"].lower() == color.lower()]


def filter_by_style(products: List[Dict[str, Any]], style: str) -> List[Dict[str, Any]]:
    """Filter products by style."""
    return [p for p in products if p["style"].lower() == style.lower()]


def pick_product(products: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Pick one product from the candidate list."""
    if not products:
        return {"error": "No matching product found"}
    return random.choice(products)


# ----------------------------
# Create agent
# ----------------------------

agent = create_agent(
    model=model,
    tools=[list_all_products, filter_by_color, filter_by_style, pick_product],
    system_prompt=(
        "You are a shopping assistant.\n"
        "To answer the user's request, you must follow this exact procedure:\n"
        "1. Call list_all_products\n"
        "2. Call filter_by_color on the result\n"
        "3. Call filter_by_style on the filtered result\n"
        "4. Call pick_product on the final candidates\n"
        "5. Then explain the recommendation briefly\n"
        "Do not skip steps."
    ),
)


# ----------------------------
# Case 1: recursion limit too small
# ----------------------------

print("=== Case 1: recursion_limit = 2 ===\n")

try:
    result_small = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Recommend me a red casual shirt."
                }
            ]
        },
        config={"recursion_limit": 2}
    )

    for m in result_small["messages"]:
        m.pretty_print()

except Exception as e:
    print("Agent failed before completing the task:")
    print(type(e).__name__, "-", e)


print("\n" + "=" * 100 + "\n")


# ----------------------------
# Case 2: recursion limit large enough
# ----------------------------

print("=== Case 2: recursion_limit = 20 ===\n")

try:
    result_ok = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Recommend me a red casual shirt."
                }
            ]
        },
        config={"recursion_limit": 20}
    )

    for m in result_ok["messages"]:
        m.pretty_print()

except Exception as e:
    print("Agent failed:")
    print(type(e).__name__, "-", e)

=== Case 1: recursion_limit = 2 ===

Agent failed before completing the task:
GraphRecursionError - Recursion limit of 2 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT


=== Case 2: recursion_limit = 20 ===

================================ Human Message =================================

Recommend me a red casual shirt.
================================== Ai Message ==================================
Tool Calls:
  list_all_products (713471093)
 Call ID: 713471093
  Args:
================================= Tool Message =================================
Name: list_all_products

[{"id": 1, "name": "Classic Red Shirt", "color": "red", "style": "casual"}, {"id": 2, "name": "Formal Blue Shirt", "color": "blue", "style": "formal"}, {"id": 3, "name": "Relaxed Green Shirt", "color": "green", "style": "casual"}, {"id": 4, "name

## 5.6 Summary

In this chapter, we explored how an AI agent is more than just an LLM—it's a coordinated system of multiple components working together.

**Key Takeaways:**

**Two Types of Components**
- **Structural components** (LLM, tools, checkpoint, memory) define what the agent *can do*
- **Runtime configuration** (thread ID, context, recursion limit) determines how the agent *behaves*

**Critical Insights**
- A raw LLM cannot use tools, remember conversations, or perform multi-step reasoning on its own
- Tool calling alone does not make an agent stateful—you need checkpoint + thread ID
- The recursion limit is a safety mechanism that prevents infinite loops but must be set appropriately for your task

**Progression Through Examples**
1. **Example 1**: Basic agent with tools but no memory—each call is independent
2. **Example 2**: Agent with checkpoint and context—enables multi-turn conversations with persistent memory
3. **Example 3**: Understanding recursion limits—tasks requiring multiple steps need adequate execution budget

**The Mental Model**

Think of building agents like assembling a bicycle:
- First, you assemble the **structural parts** (frame, wheels, brakes)
- Then during the ride, you make **runtime decisions** (which gear, when to brake)
- The structure defines capabilities; the runtime choices determine behavior

In the next chapters, we will dive deeper into **LangGraph**, the underlying framework that powers `create_agent`, and learn how to build more sophisticated multi-agent systems with custom workflows.